# Sentiment Volatility Analysis
GitHub Support Operations Analytics - Sentiment Shift Detection

This notebook identifies the "Frustration Metric" by analyzing sentiment delta between initial issue and final comment.
Helps distinguish between Product Problems vs Support Problems.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
import warnings
warnings.filterwarnings('ignore')


nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()

In [ ]:
# 1. Database Connection
engine = create_engine('mssql+pyodbc://YourServer/YourDB?driver=ODBC+Driver+17+for+SQL+Server')

In [ ]:
# 2. Fetch data (Joining Fact_Tickets with raw text stored in staging)
query = "SELECT Ticket_Key, Issue_Question, Last_Comment_Text FROM Fact_Tickets"
df = pd.read_sql(query, engine)

print(f"Loaded {len(df)} tickets for sentiment analysis")
print(df.head())

In [ ]:
# 3. Sentiment Analysis Function
def get_score(text):
    if not text or pd.isna(text):
        return 0
    return sia.polarity_scores(str(text))['compound']

In [ ]:

df['Initial_Sentiment'] = df['Issue_Question'].apply(get_score)
df['Final_Sentiment'] = df['Last_Comment_Text'].apply(get_score)

print("Sentiment scores calculated:")
print(df[['Initial_Sentiment', 'Final_Sentiment']].describe())

In [ ]:
# 5. Calculate Sentiment Volatility (Shift)
# Negative shift means user got angrier/more frustrated during the process
df['Sentiment_Shift'] = df['Final_Sentiment'] - df['Initial_Sentiment']

# "Toxic Interaction" flag: Initial sentiment was neutral/positive, but final was negative
df['is_frustrated_user'] = ((df['Initial_Sentiment'] >= 0) & (df['Sentiment_Shift'] < -0.5)).astype(int)

print("\nSentiment Shift Statistics:")
print(df['Sentiment_Shift'].describe())
print(f"\nFrustrated Users Count: {df['is_frustrated_user'].sum()}")

In [ ]:
# 6. Upload to SQL as a new Sentiment Fact Table
sentiment_output = df[['Ticket_Key', 'Initial_Sentiment', 'Final_Sentiment', 'Sentiment_Shift', 'is_frustrated_user']]

sentiment_output.to_sql(
    'Fact_Sentiment', 
    con=engine, 
    if_exists='replace', 
    index=False
)

print("Phase 2 Complete: Sentiment Data pushed to Fact_Sentiment table.")
print(f"Total records: {len(sentiment_output)}")
print(f"Frustrated users flagged: {sentiment_output['is_frustrated_user'].sum()}")

In [ ]:
# 7.  Statistics
print("\n=== SENTIMENT VOLATILITY SUMMARY ===")
print(f"Average Initial Sentiment: {df['Initial_Sentiment'].mean():.3f}")
print(f"Average Final Sentiment: {df['Final_Sentiment'].mean():.3f}")
print(f"Average Sentiment Shift: {df['Sentiment_Shift'].mean():.3f}")
print(f"\nPositive Shift (User Got Happier): {(df['Sentiment_Shift'] > 0.1).sum()} tickets")
print(f"Negative Shift (User Got Frustrated): {(df['Sentiment_Shift'] < -0.1).sum()} tickets")
print(f"Stable Sentiment: {((df['Sentiment_Shift'] >= -0.1) & (df['Sentiment_Shift'] <= 0.1)).sum()} tickets")